In [ ]:
import numpy as np

def _summarize_object(x, max_items=8):
    """Try to summarize object arrays without spamming output."""
    try:
        # common case: 0-d array containing dict
        if isinstance(x, np.ndarray) and x.dtype == object:
            if x.shape == ():  # scalar object
                obj = x.item()
                return _summarize_object(obj, max_items=max_items)
            # array of objects
            flat = list(x.ravel()[:max_items])
            return f"object array, first {len(flat)} items types: {[type(v).__name__ for v in flat]}"
        # python object
        if isinstance(x, dict):
            keys = list(x.keys())
            keys_preview = keys[:max_items]
            return f"dict with {len(keys)} keys: {keys_preview}{' ...' if len(keys)>max_items else ''}"
        if isinstance(x, (list, tuple)):
            return f"{type(x).__name__} of len {len(x)}; first types: {[type(v).__name__ for v in x[:max_items]]}"
        if isinstance(x, str):
            return f"str(len={len(x)}): {x[:120]}{' ...' if len(x)>120 else ''}"
        return f"{type(x).__name__}: {repr(x)[:200]}{' ...' if len(repr(x))>200 else ''}"
    except Exception as e:
        return f"(could not summarize object: {e})"

def inspect_npz(path):
    print("=" * 100)
    print(f"FILE: {path}")
    print("-" * 100)

    d = np.load(path, allow_pickle=True)
    print("Keys found:", d.files)
    print()

    for k in d.files:
        arr = d[k]
        print(f"Key: '{k}'")
        print(f"  dtype : {arr.dtype}")
        print(f"  shape : {arr.shape}")

        if arr.dtype == object:
            # show a compact summary of what is inside
            try:
                summary = _summarize_object(arr)
                print(f"  content summary: {summary}")
                # if it's a scalar dict, also preview a few items
                if arr.shape == () and isinstance(arr.item(), dict):
                    md = arr.item()
                    preview_keys = list(md.keys())[:8]
                    for kk in preview_keys:
                        vv = md[kk]
                        print(f"    - {kk}: {type(vv).__name__}")
            except Exception as e:
                print(f"  (object inspection error: {e})")

        elif np.issubdtype(arr.dtype, np.number):
            try:
                # numeric stats
                if np.issubdtype(arr.dtype, np.floating):
                    n_nan = np.isnan(arr).sum()
                    print(f"  min   : {np.nanmin(arr):.5g}")
                    print(f"  max   : {np.nanmax(arr):.5g}")
                    print(f"  NaNs  : {n_nan}")
                else:
                    print(f"  min   : {arr.min()}")
                    print(f"  max   : {arr.max()}")
            except Exception as e:
                print(f"  stats error: {e}")

        else:
            print("  (non-numeric data)")

        print("-" * 60)

    print("=" * 100)
    print()


# ------------------------------------------------------------------
# EDIT THESE PATHS
# ------------------------------------------------------------------

wind_file = "/path/to/SAR_sea_ice_dataset/MEAN_CARRA_WIND_8steps/region-27_0-82_951-35_52-83_8/2014/11/07/20141107T0807_mean_wind_8steps_future.npz"

past_drift_file = "/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs_velocity/HV/region-27_0-82_951-35_52-83_8/2014/11/07/20141106T0905__20141107T0807_past.npz"

future_drift_file = "/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs_velocity/HV/region-27_0-82_951-35_52-83_8/2014/11/07/20141107T0807__20141108T0848_future.npz"


# ------------------------------------------------------------------
# RUN INSPECTION
# ------------------------------------------------------------------

inspect_npz(wind_file)
inspect_npz(past_drift_file)
inspect_npz(future_drift_file)


In [ ]:
import os, re, json, glob
from pathlib import Path
from typing import Optional

TS_RE = re.compile(r"(\d{8}T\d{4})")

def t_from_wind(path: str) -> Optional[str]:
    m = TS_RE.search(Path(path).name)
    return m.group(1) if m else None

def t_from_past(path: str) -> Optional[str]:
    # ".../A__B_past.npz" -> t = B
    name = Path(path).name
    if "__" not in name:
        return None
    right = name.split("__", 1)[1]
    m = TS_RE.search(right)
    return m.group(1) if m else None

def t_from_future(path: str) -> Optional[str]:
    # ".../A__B_future.npz" -> t = A
    name = Path(path).name
    if "__" not in name:
        return None
    left = name.split("__", 1)[0]
    m = TS_RE.search(left)
    return m.group(1) if m else None

def build_index_jsonl(
    wind_region_root: str,
    drift_region_root: str,
    out_jsonl: str,
    require_same_shape: bool = True,
    verbose: bool = True,
):
    wind_files  = glob.glob(os.path.join(wind_region_root,  "**", "*_future.npz"), recursive=True)
    past_files  = glob.glob(os.path.join(drift_region_root, "**", "*_past.npz"),   recursive=True)
    fut_files   = glob.glob(os.path.join(drift_region_root, "**", "*_future.npz"), recursive=True)

    wind_by_t = {t_from_wind(p): p for p in wind_files if t_from_wind(p)}
    past_by_t = {t_from_past(p): p for p in past_files if t_from_past(p)}
    fut_by_t  = {t_from_future(p): p for p in fut_files if t_from_future(p)}

    ts = sorted(set(wind_by_t) & set(past_by_t) & set(fut_by_t))

    n_written = 0
    skipped_shape = 0

    with open(out_jsonl, "w") as f:
        for i, t in enumerate(ts):
            row = {
                "id": i,
                "t": t,
                "wind_path": wind_by_t[t],
                "past_path": past_by_t[t],
                "future_path": fut_by_t[t],
            }

            if require_same_shape:
                import numpy as np
                w = np.load(row["wind_path"], allow_pickle=True)
                p = np.load(row["past_path"], allow_pickle=True)
                fu = np.load(row["future_path"], allow_pickle=True)

                shapes = [
                    w["u10_mean"].shape, w["v10_mean"].shape,
                    p["u"].shape, p["v"].shape,
                    fu["u"].shape, fu["v"].shape
                ]
                if len(set(shapes)) != 1:
                    skipped_shape += 1
                    continue

            f.write(json.dumps(row) + "\n")
            n_written += 1

    if verbose:
        print(f"Wind files : {len(wind_files)}")
        print(f"Past files : {len(past_files)}")
        print(f"Future files: {len(fut_files)}")
        print(f"Matched t  : {len(ts)}")
        print(f"Wrote      : {n_written} -> {out_jsonl}")
        print(f"Skipped (shape mismatch): {skipped_shape}")


In [ ]:
# build_index_jsonl(
#     wind_region_root="/path/to/SAR_sea_ice_dataset/MEAN_CARRA_WIND_8steps/region-27_0-82_951-35_52-83_8",
#     drift_region_root="/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-27_0-82_951-35_52-83_8",
#     out_jsonl="index_region-27_0-82_951-35_52-83_8.jsonl",
# )


In [ ]:
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class DriftWindDataset(Dataset):
    """
    X = [past_u, past_v, wind_u10_mean, wind_v10_mean]  -> (4,H,W)
    Y = [future_u, future_v]                            -> (2,H,W)
    """
    def __init__(self, index_jsonl, include_wspd=False, return_meta=False, transform=None):
        self.rows = [json.loads(line) for line in open(index_jsonl, "r")]
        self.include_wspd = include_wspd
        self.return_meta = return_meta
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        # load with allow_pickle=True because of attrs/meta dicts
        w = np.load(r["wind_path"], allow_pickle=True)
        p = np.load(r["past_path"], allow_pickle=True)
        f = np.load(r["future_path"], allow_pickle=True)

        past_u = p["u"].astype(np.float32)
        past_v = p["v"].astype(np.float32)
        wind_u = w["u10_mean"].astype(np.float32)
        wind_v = w["v10_mean"].astype(np.float32)

        x_list = [past_u, past_v, wind_u, wind_v]

        if self.include_wspd:
            wspd = w["wspd_mean"].astype(np.float32)
            x_list.append(wspd)

        x = np.stack(x_list, axis=0)  # (C,H,W)

        fut_u = f["u"].astype(np.float32)
        fut_v = f["v"].astype(np.float32)
        y = np.stack([fut_u, fut_v], axis=0)  # (2,H,W)

        if self.transform is not None:
            x, y = self.transform(x, y)

        out = {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "t": r["t"],
            "id": r["id"],
        }

        if self.return_meta:
            out["wind_attrs"] = w["attrs"].item() if "attrs" in w else None
            out["drift_meta_past"] = p["meta"].item() if "meta" in p else None
            out["drift_meta_future"] = f["meta"].item() if "meta" in f else None

        return out

# ---- quick sanity check ----
ds = DriftWindDataset("index_region-27_0-82_951-35_52-83_8.jsonl", include_wspd=False, return_meta=True)
sample = ds[0]
print(sample["x"].shape, sample["y"].shape, sample["t"])
print("wind attrs keys:", list(sample["wind_attrs"].keys()))
print("past meta keys:", list(sample["drift_meta_past"].keys()))

# ---- dataloader ----
dl = DataLoader(ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=True)
batch = next(iter(dl))
print("Batch x:", batch["x"].shape, "Batch y:", batch["y"].shape)


In [ ]:
import torch

@torch.no_grad()
def compute_channel_stats(ds, n_samples=1500, seed=0):
    # compute mean/std over a subset for speed; increase later
    g = torch.Generator().manual_seed(seed)
    idxs = torch.randperm(len(ds), generator=g)[:min(n_samples, len(ds))].tolist()

    # x has shape (C,H,W)
    c = ds[0]["x"].shape[0]
    mean = torch.zeros(c)
    m2 = torch.zeros(c)
    count = 0

    for i in idxs:
        x = ds[i]["x"].float()  # (C,H,W)
        x = x.view(c, -1)

        # mean over pixels per channel for this sample
        sample_mean = x.mean(dim=1)
        sample_var = x.var(dim=1, unbiased=False)

        # combine per-sample stats into global (approx)
        # treat each sample as having same number of pixels
        count += 1
        delta = sample_mean - mean
        mean += delta / count
        m2 += sample_var  # approximate pooling; good enough to start

    std = torch.sqrt(m2 / max(count, 1)).clamp_min(1e-6)
    return mean, std

x_mean, x_std = compute_channel_stats(ds, n_samples=200, seed=0)
print("x_mean:", x_mean)
print("x_std :", x_std)


In [ ]:
@torch.no_grad()
def compute_y_channel_stats(ds, n_samples=500, seed=0):
    g = torch.Generator().manual_seed(seed)
    idxs = torch.randperm(len(ds), generator=g)[:min(n_samples, len(ds))].tolist()

    mean = torch.zeros(2)
    m2 = torch.zeros(2)
    count = 0

    for i in idxs:
        y = ds[i]["y"].float().view(2, -1)

        sample_mean = y.mean(dim=1)
        sample_var  = y.var(dim=1, unbiased=False)

        count += 1
        delta = sample_mean - mean
        mean += delta / count
        m2 += sample_var

    std = torch.sqrt(m2 / max(count, 1)).clamp_min(1e-6)
    return mean, std

y_mean, y_std = compute_y_channel_stats(ds, n_samples=200, seed=0)
print("y_mean:", y_mean)
print("y_std :", y_std)


In [ ]:
class NormalizeXY:
    def __init__(self, x_mean, x_std, y_mean=None, y_std=None):
        self.x_mean = x_mean.view(-1, 1, 1)
        self.x_std = x_std.view(-1, 1, 1)
        self.y_mean = None if y_mean is None else y_mean.view(-1, 1, 1)
        self.y_std  = None if y_std  is None else y_std.view(-1, 1, 1)

    def __call__(self, x, y):
        # x,y are numpy arrays (C,H,W)
        x = (x - self.x_mean.numpy()) / self.x_std.numpy()
        if self.y_mean is not None and self.y_std is not None:
            y = (y - self.y_mean.numpy()) / self.y_std.numpy()
        return x, y

norm = NormalizeXY(x_mean, x_std, y_mean, y_std)

# ---- quick sanity check ----
ds = DriftWindDataset("index_region-27_0-82_951-35_52-83_8.jsonl", include_wspd=False, return_meta=True, transform=norm)
sample = ds[0]
print(sample["x"].shape, sample["y"].shape, sample["t"])
print("wind attrs keys:", list(sample["wind_attrs"].keys()))
print("past meta keys:", list(sample["drift_meta_past"].keys()))

# ---- dataloader ----
dl = DataLoader(ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
batch = next(iter(dl))
print("Batch x:", batch["x"].shape, "Batch y:", batch["y"].shape)